# Coco Crepe — 01 Bronze Sources

Carga técnica de las fuentes Raw hacia Bronze. No aplica transformaciones de negocio.

In [0]:
GROUP = "g203"

SALES_DATA_PRODUCT = "sales_summary"
INVENTORY_DATA_PRODUCT = "inventory_status"
PRODUCT_DATA_PRODUCT = "product_master"

# Completa estos valores solo si la detección automática no encuentra
# exactamente un catálogo por Data Product.
SALES_CATALOG_MANUAL = None
INVENTORY_CATALOG_MANUAL = "g203_inv_inventory_status"
PRODUCT_CATALOG_MANUAL = None

def resolve_catalog(data_product_name, manual_catalog=None):
    if manual_catalog:
        return manual_catalog

    catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
    target = data_product_name.lower()

    preferred = [
        catalog for catalog in catalogs
        if GROUP.lower() in catalog.lower()
        and target in catalog.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    matches = [
        catalog for catalog in catalogs
        if target in catalog.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"No se pudo identificar un catálogo único para '{data_product_name}'. "
        f"Catálogos visibles: {catalogs}. "
        "Completa la variable *_CATALOG_MANUAL correspondiente."
    )

SALES_CATALOG = resolve_catalog(
    SALES_DATA_PRODUCT,
    SALES_CATALOG_MANUAL
)

INVENTORY_CATALOG = resolve_catalog(
    INVENTORY_DATA_PRODUCT,
    INVENTORY_CATALOG_MANUAL
)

PRODUCT_CATALOG = resolve_catalog(
    PRODUCT_DATA_PRODUCT,
    PRODUCT_CATALOG_MANUAL
)

print(f"Sales catalog: {SALES_CATALOG}")
print(f"Inventory catalog: {INVENTORY_CATALOG}")
print(f"Product catalog: {PRODUCT_CATALOG}")

In [0]:
BASE_PATH = (
    "abfss://datalake@stdemdsai.dfs.core.windows.net/"
    "raw/airflow2/G3/creperia"
)

product_bronze = f"{PRODUCT_CATALOG}.bronze.products"

sales_customers = f"{SALES_CATALOG}.bronze.customers"
sales_orders = f"{SALES_CATALOG}.bronze.orders"
sales_order_items = f"{SALES_CATALOG}.bronze.order_items"

inventory_source = f"{INVENTORY_CATALOG}.bronze.inventory"
inventory_suppliers = f"{INVENTORY_CATALOG}.bronze.suppliers"


In [0]:
# Product Master
spark.sql(f"""
CREATE OR REPLACE TABLE {product_bronze} AS
SELECT
    *,
    current_timestamp() AS inserted_at
FROM read_files(
    '{BASE_PATH}/products/load_date=*/products.csv',
    format => 'csv',
    header => true,
    schema => 'product_id INT, product_name STRING, category STRING, price DOUBLE'
)
""")

# Sales Summary
spark.sql(f"""
CREATE OR REPLACE TABLE {sales_customers} AS
SELECT
    *,
    current_timestamp() AS inserted_at
FROM read_files(
    '{BASE_PATH}/customers/load_date=*/customers.csv',
    format => 'csv',
    header => true,
    schema => 'customer_id INT, customer_name STRING, district STRING, registration_date DATE'
)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {sales_orders} AS
SELECT
    *,
    current_timestamp() AS inserted_at
FROM read_files(
    '{BASE_PATH}/orders/load_date=*/orders.csv',
    format => 'csv',
    header => true,
    schema => 'order_id INT, order_date DATE, customer_id INT, total_amount DOUBLE, payment_method STRING'
)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {sales_order_items} AS
SELECT
    *,
    current_timestamp() AS inserted_at
FROM read_files(
    '{BASE_PATH}/order_items/load_date=*/order_items.csv',
    format => 'csv',
    header => true,
    schema => 'item_id INT, order_id INT, product_id INT, quantity INT, unit_price DOUBLE'
)
""")

# Inventory Status
spark.sql(f"""
CREATE OR REPLACE TABLE {inventory_source} AS
SELECT
    *,
    current_timestamp() AS inserted_at
FROM read_files(
    '{BASE_PATH}/inventory/load_date=*/inventory.csv',
    format => 'csv',
    header => true,
    schema => 'product_id INT, stock INT, supplier_id INT, last_update DATE'
)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {inventory_suppliers} AS
SELECT
    *,
    current_timestamp() AS inserted_at
FROM read_files(
    '{BASE_PATH}/suppliers/load_date=*/suppliers.csv',
    format => 'csv',
    header => true,
    schema => 'supplier_id INT, supplier_name STRING, city STRING'
)
""")

In [0]:
technical_checks = [
    (product_bronze, "product_id"),
    (sales_customers, "customer_id"),
    (sales_orders, "order_id"),
    (sales_order_items, "item_id"),
    (inventory_source, "product_id"),
    (inventory_suppliers, "supplier_id"),
]

for table_name, key_column in technical_checks:
    spark.sql(f"""
        SELECT
            '{table_name}' AS table_name,
            COUNT(*) AS record_count,
            SUM(CASE WHEN {key_column} IS NULL THEN 1 ELSE 0 END) AS null_keys
        FROM {table_name}
    """).display()